# `ClassBase`

`nematics3d.ClassBase` is the common structured-object base used by many `Nematics3D` classes. Ordinary users rarely instantiate it directly. Instead, concrete objects inherit a shared vocabulary for naming attributes, inspecting their public interface, carrying a stable object name, attaching side data, and recording semantic relations to other objects.

This tutorial is a reference-oriented introduction to that common vocabulary. The most important idea is that names in a `ClassBase` object are deliberately structured: prefixes tell you what role an attribute or method plays. Once those conventions are familiar, the inspection methods beginning with `show_` provide a consistent way to understand an unfamiliar `Nematics3D` object.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell defines a small `ClassBase` subclass containing examples of the major attribute categories. The subclass exists only so that the rest of the tutorial can demonstrate the `ClassBase` interface in isolation.


In [ ]:
from nematics3d.core.class_base import AttrDef, ClassBase


class ExampleObject(ClassBase):
    """Small object used to demonstrate the ClassBase public interface."""

    __attr_defs__ = {
        "raw_value": AttrDef("Primary input value.", kind="raw"),
        "state_mode": AttrDef("Current runtime mode.", kind="state"),
        "default_scale": AttrDef("Managed default scale.", kind="default"),
        "calc_double": AttrDef("Twice the input value.", kind="calc"),
        "entity_child": AttrDef("Generated child object.", kind="entity"),
        "partner": AttrDef("A related example object.", kind="relation"),
        "label": AttrDef("A writable Python property.", kind="property", is_public_settable=True),
        "impl_cache": AttrDef("Internal cache.", kind="impl"),
    }

    __slots__ = ("raw_value", "state_mode", "default_scale", "calc_double", "entity_child", "_label", "impl_cache")

    def __init__(self, value=2, name="example"):
        super().__init__(name=name, name_replace="example")
        object.__setattr__(self, "raw_value", value)
        object.__setattr__(self, "state_mode", "ready")
        object.__setattr__(self, "default_scale", 1.0)
        object.__setattr__(self, "calc_double", 2 * value)
        object.__setattr__(self, "entity_child", None)
        object.__setattr__(self, "_label", "demo")
        object.__setattr__(self, "impl_cache", {})

    @property
    def label(self):
        return self._label

    @label.setter
    def label(self, value):
        object.__setattr__(self, "_label", str(value))


obj = ExampleObject()


## Prefix conventions

The naming convention is part of the `ClassBase` object protocol. A prefix is not merely descriptive decoration: for managed attributes, `ClassBase` checks that the declared semantic kind agrees with the name. For example, a field declared with `kind="raw"` must be named `raw_...`, and a field named `calc_...` must represent calculated output.

The main attribute prefixes are:

| Prefix | Semantic kind | Meaning | Public behavior |
| --- | --- | --- | --- |
| `raw_...` | `raw` | Canonical stored input or base data | Readable and writable; also exposes a shorter readable/writable alias |
| `state_...` | `state` | Writable runtime state | Readable and writable under its full name |
| `default_...` | `default` | Managed default-layer input | Readable and writable under its full name |
| `calc_...` | `calc` | Calculated result | Readable, normally not publicly writable |
| `entity_...` | `entity` | Calculated or generated object-valued result | Readable, normally not publicly writable |
| `impl_...` | `impl` | Internal implementation state | Not part of the ordinary public inspection surface |

Two additional managed kinds deliberately have no required prefix:

- a `relation` is a semantic link to another object, such as `owner`, `registry`, or `partner`;
- a `property` is backed by a normal Python `@property` and may be read-only or explicitly public-settable.


### `raw_` has a public alias

`raw_` is special because its prefix may be omitted on the public surface. `raw_value` is the canonical managed name, while `value` is the convenient alias. Both refer to the same stored input.


In [ ]:
print(obj.raw_value)
print(obj.value)

obj.value = 5
print(obj.raw_value)


### Method prefixes

The same principle is used for methods. Public inspection methods begin with `show_`, while public operations that change or act through an object commonly begin with `act_`. Internal reusable implementation helpers begin with `_helper_`.

| Prefix | Meaning | Intended audience |
| --- | --- | --- |
| `show_...` | Inspect, display, or explain | Users |
| `act_...` | Perform an operation | Users |
| `_helper_...` | Internal implementation helper | Developers |

This makes editor and notebook autocomplete useful as API discovery. Typing `obj.show_` narrows the interface to questions you can ask the object; typing `obj.act_` narrows it to actions you can request.


## The `show_` inspection interface

`ClassBase` provides six general inspection methods. Together they answer three kinds of questions: what the object is, what attributes it exposes, and how it is related to other objects. All of them print through the `Nematics3D` logging interface by default; passing `is_return=True` returns the formatted text instead.


### `show_doc()`: what is this object?

`show_doc()` displays the docstring of the object's concrete class. It is the fastest general-purpose way to ask an unfamiliar `ClassBase` descendant what it represents.


In [ ]:
obj.show_doc()


### `show_readable_attrs()`: what can I read?

`show_readable_attrs()` lists the registered public attributes and their descriptions. Internal `impl_` fields are intentionally omitted. Dynamic extra attributes, introduced later in this tutorial, are included.

Set `is_desc=False` when only the names are needed.


In [ ]:
obj.show_readable_attrs()


### `show_modifiable_attrs()`: what am I allowed to change?

`show_modifiable_attrs()` lists the currently public-settable surface. For plain `ClassBase` objects this normally includes `raw_`, `state_`, and `default_` fields, explicitly writable properties, and dynamic extra attributes. Protected fields and fixed core fields are omitted because they cannot currently be changed through public assignment.

As with readable attributes, `is_desc=False` suppresses the descriptions.


In [ ]:
obj.show_modifiable_attrs()


### `show_attr_doc()`: what does one attribute mean?

When you already know the attribute name, `show_attr_doc()` returns only its registered documentation. A `raw_` field may be addressed by either its canonical name or its shorter alias.


In [ ]:
obj.show_attr_doc("value")
obj.show_attr_doc("calc_double")


### `show_attr_info()`: give me the complete status of one attribute

`show_attr_info()` is the more detailed counterpart of `show_attr_doc()`. It reports the canonical name, semantic kind, alias when applicable, whether the field is currently modifiable or protected, a bounded representation of its current value, and its documentation.


In [ ]:
obj.show_attr_info("value")


### `show_relations()`: what objects am I connected to?

`show_relations()` lists relations that are currently bound, together with their descriptions and current targets. Declared relations whose target is `None` are omitted.


In [ ]:
other = ExampleObject(name="other")
obj.act_bind_relation_base("partner", other)
obj.show_relations()


### `show_relation_tree()`: how is the relation graph organized?

`show_relation_tree()` recursively displays the relation graph beginning at the current object. `depth` controls how many relation levels are traversed; the default is `2`. Set `is_include_none=True` to show declared but currently unbound relations as well. Cycles are detected and marked as already visited rather than traversed indefinitely.


In [ ]:
other.act_bind_relation_base("partner", obj)
obj.show_relation_tree(depth=2, is_include_none=True)


### Quick reference for `show_`

| Method | Main question | Important options |
| --- | --- | --- |
| `show_doc()` | What is this object? | `is_return` |
| `show_readable_attrs()` | What can I read? | `is_desc`, `is_return` |
| `show_modifiable_attrs()` | What can I change now? | `is_desc`, `is_return` |
| `show_attr_doc(name)` | What does this field mean? | `is_return` |
| `show_attr_info(name)` | What is the complete status of this field? | `is_return` |
| `show_relations()` | What objects are currently related? | `is_return` |
| `show_relation_tree()` | How is the relation graph structured? | `depth`, `is_include_none`, `is_return` |

Although the methods share the `show_` prefix, they are not redundant. A useful progression for an unfamiliar object is `show_doc()` → `show_readable_attrs()` → `show_attr_info(...)`, with the relation methods used when the object participates in a larger object structure.


## Object identity: `name` and `raw_name`

Every `ClassBase` instance has a string identifier. Its canonical stored field is `raw_name`, while `name` is the ordinary public alias. This follows the same `raw_` alias rule introduced earlier.


In [ ]:
print(obj.raw_name)
print(obj.name)


### Renaming an object

Use `act_set_name()` when you want to rename an object explicitly. Public assignment to `name` or `raw_name` is routed through the same name-validation path. If the object belongs to a compatible registry, the registry may adjust the requested name to preserve uniqueness.


In [ ]:
obj.act_set_name("renamed example")
print(obj.name)


### `str()` and `repr()`

The default `ClassBase` representation combines the concrete class name with the object's `name`. `ClassBase` defines `__repr__()` directly; unless a subclass defines a separate `__str__()`, normal Python string conversion follows the inherited representation behavior. Concrete subclasses are free to provide richer display conventions.


In [ ]:
print(repr(obj))
print(str(obj))


## Reading the attribute model

The prefix convention describes the public vocabulary; the semantic kinds describe what `ClassBase` does with that vocabulary. The distinction matters most for mutability.

- `raw`, `state`, and `default` fields are public-settable by default.
- `calc` and `entity` fields are public readable outputs rather than assignment targets.
- `property` fields are writable only when the class explicitly declares them public-settable and provides the corresponding property behavior.
- `relation` fields are managed object links and are changed through relation operations rather than ordinary assignment.
- `impl` fields are implementation state and are intentionally excluded from ordinary public inspection.
- `extra` attributes are instance-local side data registered at runtime.

This is why `show_readable_attrs()` and `show_modifiable_attrs()` are different questions: an object may expose many calculated results that are useful to read but should never be assigned by the user.


### Temporarily protecting writable attributes

A public-settable field can be marked protected with `act_register_protected_attr()`. Protected fields disappear from `show_modifiable_attrs()` and reject public assignment until protection is removed with `act_unregister_protected_attr()`. The shorter alias of a `raw_` field may be used when registering protection.


In [ ]:
obj.act_register_protected_attr("value")
obj.show_modifiable_attrs(is_desc=False)
obj.act_unregister_protected_attr("value")


### Fixed objects

A concrete class may initialize `ClassBase` with `is_fixed=True`. In that mode its core `raw_` and `state_` fields are frozen after initialization because changing them could invalidate too much dependent state. The object name remains independently manageable, and side-data extra attributes are not part of the frozen core. For a fixed object, `show_modifiable_attrs()` reflects the reduced writable surface.


## Dynamic extra attributes

`act_add_attr()` attaches documented, instance-local side data that is not part of the class's scientific or structural schema. Typical examples are an analysis note, a sample identifier, or provenance information. Extra attributes become visible to the same inspection interface as declared fields.


In [ ]:
obj.act_add_attr("note", "A user annotation for this object.", default="candidate A")
print(obj.note)
obj.show_attr_info("note")

old_note = obj.act_remove_attr("note")
print(old_note)


Extra attributes intentionally cannot use reserved semantic prefixes such as `raw_`, `calc_`, or `impl_`, and their names cannot collide with existing readable fields or class-level public surfaces. They are side data, not a way to add a new semantic field to a class at runtime.

An optional validator can be supplied to `act_add_attr()` when the side data itself needs normalization or validation.


## Relations

Relations are named one-to-one semantic links between objects. `ClassBase` itself declares `owner` and `registry`; subclasses may declare additional relations. A relation must be part of the class schema before it can be bound.

The low-level public operations are `act_bind_relation_base()` and `act_unbind_relation_base()`. A concrete class may provide a more domain-specific operation on top of them, such as a registry's registration methods.


In [ ]:
obj.act_bind_relation_base("partner", other)
print(obj.partner is other)

obj.act_unbind_relation_base("partner")
print(obj.partner)


### Weak and strong relations

A declared relation has a default weak-reference policy. `ClassBase` relations default to weak references unless the declaration says otherwise. A weak relation does not keep its target alive merely because the relation exists; a strong relation does. `act_bind_relation_base(..., is_weak=...)` can override the declared default for a particular binding.

The distinction affects object lifetime, not the ordinary read syntax: in either case, reading `obj.partner` returns the target object while it exists.


## Public-interface summary

The most important `ClassBase` conventions can be summarized as follows:

1. Read attribute prefixes semantically: `raw_`, `state_`, `default_`, `calc_`, `entity_`, and `impl_` describe the role of the value.
2. Use `show_` methods to inspect an unfamiliar object rather than starting from implementation details.
3. Every object has a common `name` / `raw_name` identity interface.
4. Readable and modifiable surfaces are deliberately different; calculated output can be public without being writable.
5. Extra attributes hold instance-local side data, while relations express semantic links to other objects.

These conventions form the object vocabulary inherited by higher-level classes such as `RegistryBase` and `HostBase`.


## Developer reference: declaring `ClassBase` fields

**This section is intended for developers. Regular users can safely skip it.** A subclass declares its managed fields through class-level `__attr_defs__` entries containing frozen `AttrDef` objects. Parent declarations are automatically merged through the method resolution order, so a subclass declares only its own additions or intentional overrides.


### `AttrDef`

An `AttrDef` records the static contract of one managed field:

- `doc` is its public description;
- `kind` selects its semantic category;
- `validator` optionally normalizes or validates public assignment;
- `is_public_settable` explicitly controls property writability;
- `is_weak_by_default` controls the default lifetime semantics of a relation;
- `is_reapply_opts_after_raw` is metadata consumed by `HostBase`, not by plain `ClassBase`.

Prefix and kind are checked in both directions when the subclass is created. This is intentional: a developer should be able to infer the semantic role of a managed field from its name.


### Minimal declaration pattern

A subclass can add fields without manually copying the parent's schema:

```python
class MyObject(ClassBase):
    __attr_defs__ = {
        "raw_data": AttrDef("Primary input data.", kind="raw"),
        "calc_result": AttrDef("Calculated result.", kind="calc"),
    }
```

The inherited `raw_name`, `owner`, `registry`, and implementation declarations are merged automatically. Core managed values should still be stored in the corresponding slot-backed attributes; `__attr_defs__` describes the schema rather than storing per-instance values.
